# web_agent — Kaggle driver (hybrid)

Stable core (`labels`, `config`, `dataset`, `model`, `loss`, `metrics`) lives in the
`web_agent` package, cloned from GitHub. Orchestration + analysis (training loop,
eval, plots, error inspection, result images) lives **here in the notebook** so you
can watch every step.

Workflow: edit package in IDE → push to GitHub `Code` → **Pull** here → re-run.

In [ ]:
# 1. Clone the package from GitHub, install deps, make it importable (re-run safe)
import os, sys
REPO = "https://github.com/Kiyas-Mahmud/webagent.git"
ROOT = "/kaggle/working/webagent"
if not os.path.isdir(ROOT):
    !git clone -b Code {REPO} {ROOT}
else:
    !cd {ROOT} && git pull --ff-only
%cd {ROOT}
!pip install -e . -q
# Editable install (PEP 660) doesn't reach the live kernel without a restart,
# so add src/ to sys.path directly — imports work immediately.
if f"{ROOT}/src" not in sys.path:
    sys.path.insert(0, f"{ROOT}/src")
import web_agent
print("web_agent ready ->", web_agent.__file__)

In [ ]:
# 2. Confirm GPU + dataset path, then verify the dataset (Phase 1, in-kernel)
import os
from pathlib import Path
!nvidia-smi -L
print("input dirs:", os.listdir("/kaggle/input"))

DATA_PATH = "/kaggle/input/datasets/kiyasmahmud/thesisdata/FinalData"
assert os.path.isdir(DATA_PATH), f"fix DATA_PATH; not found: {DATA_PATH}"

from web_agent.data.verify_dataset import verify
report = verify(Path(DATA_PATH))
print(report.text())
assert report.ok, "dataset verification FAILED — read the report above"

In [4]:
# 3. Phase 2 — build a DataLoader and pull ONE batch (Y1: SigLIP + RoBERTa)
from transformers import AutoProcessor, AutoTokenizer
from web_agent.config import load_config
from web_agent.utils.seed import set_seed
from web_agent.data.dataloader import load_split, build_dataloader

set_seed(42)
cfg = load_config("configs/backbones/y1_siglip_roberta.yaml")
cfg["data"]["root"] = DATA_PATH            # point at the Kaggle mount
cfg["data"]["num_workers"] = 2

processor = AutoProcessor.from_pretrained(cfg["backbone"]["vision_encoder"])
tokenizer = AutoTokenizer.from_pretrained(cfg["backbone"]["text_encoder"])

train = load_split(cfg, "train")
loader = build_dataloader(cfg, mode="full_labels", records=train,
                          processor=processor, tokenizer=tokenizer,
                          limit=cfg["stages"]["smoke"], batch_size=8)

batch = next(iter(loader))
print("batch keys:", list(batch.keys()))
for k, v in batch.items():
    print(f"  {k:18} {tuple(v.shape)}  {v.dtype}")

ModuleNotFoundError: No module named 'web_agent'

In [ ]:
# 4. Visual sanity-check — show 4 raw screenshots with their decoded labels
import matplotlib.pyplot as plt
from PIL import Image
from web_agent.labels import (
    EXECUTION_OUTCOME_INV, FAILURE_TYPE_INV, ACTION_TYPE_INV, RECOVERY_STRATEGY_INV,
)

rows = loader.dataset.records[:4]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, rec in zip(axes, rows):
    ax.imshow(Image.open(f"{DATA_PATH}/{rec['state_before']}").convert("RGB"))
    ax.axis("off")
    ax.set_title(
        f"{rec['execution_outcome']} / {rec['failure_type']}\n"
        f"act={rec['action_type']}  rec={rec['recovery_strategy']}\n"
        f"conf={rec['agent_confidence_before']:.2f}  bbox={'Y' if rec['action_target_bbox'] else 'N'}",
        fontsize=9,
    )
plt.tight_layout(); plt.show()

# Confirm encoded labels match the strings above
print("encoded label_outcome :", batch["label_outcome"][:4].tolist())
print("encoded label_action  :", batch["label_action"][:4].tolist())
print("bbox_mask             :", batch["bbox_mask"][:4].squeeze(-1).tolist())